In [2]:
import pandas as pd
import csv

In [4]:
df=pd.read_csv("RetailCleared.csv")

C:\Users\user\AppData\Local\Temp\ipykernel_13928\1073363946.py:1: DtypeWarning: Columns (0: invoice) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("RetailCleared.csv")


In [10]:
df.head()

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,rownum
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,1
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,1
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,1
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,1
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,1


In [6]:
df.tail()

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,rownum
1033031,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom,1
1033032,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,1
1033033,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom,1
1033034,C581569,20979,36 PENCILS TUBE RED RETROSPOT,-5,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,1
1033035,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,1


In [11]:
df['invoicedate']=pd.to_datetime(df['invoicedate'])

In [13]:
df.shape

(1033036, 9)

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   invoice      1033036 non-null  object        
 1   stockcode    1033036 non-null  str           
 2   description  1028761 non-null  str           
 3   quantity     1033036 non-null  int64         
 4   invoicedate  1033036 non-null  datetime64[us]
 5   price        1033036 non-null  float64       
 6   customerid   797885 non-null   float64       
 7   country      1033036 non-null  str           
 8   rownum       1033036 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), object(1), str(3)
memory usage: 70.9+ MB


In [15]:
#Filter and Calculate Recency, Frequency, Monetary
rfm_data=df[(df['customerid'].notna()) & (df['quantity']>0)].copy()
snapshot_date=pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('customerid').agg(
    Recency=('invoicedate', lambda x: (snapshot_date-x.max()).days),
    Frequency=('invoice','nunique'),
    Monetary=('quantity', lambda x: (x * rfm_data.loc[x.index,'price']).sum())
).reset_index()

In [16]:
#Score Customers and Assign Business Segments
rfm['R_Score']=pd.qcut(rfm['Recency'],5,labels=[5,4,3,2,1]).astype(int)
rfm['F_Score']=pd.qcut(rfm['Frequency'].rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)
rfm['M_Score']=pd.qcut(rfm['Monetary'],5,labels=[1,2,3,4,5]).astype(int)

def segment_customer(row):
    if row['R_Score']>=4 and row ['F_Score']>=4 and row['M_Score']>=4:
        return 'Champions'
    elif row['R_Score']>=3 and row ['F_Score']>=3:
        return 'Loyal'
    elif row['R_Score']<=2 and row ['F_Score']>=3:
        return 'At Risk'
    elif row['R_Score']>=4 and row ['F_Score']<=2:
        return 'New '
    elif row['R_Score']==3 and row ['F_Score']<=2:
        return 'Need Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90

In [21]:
# Merge RFM Result Back onto Transaction and Export]
df = df[(df['customerid'].notna()) & (df['quantity']>0)].copy()
df['InvoiceMonth']=df['invoicedate'].dt.to_period('M').dt.to_timestamp()
df['Revenue']=df['quantity']*df['price']

df_final=df.merge(
    rfm[['customerid','Recency','Frequency','Monetary','R_Score','F_Score','M_Score','Segment','IsChurned']],
    on='customerid',how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)